In [1]:
from collections import namedtuple
from enum import IntEnum
import bz2
from collections import Counter, namedtuple
from datetime import datetime
from io import StringIO
import numpy
import os
import pandas
from pathlib import Path
import re
from subprocess import run, PIPE
import sys
import zoneinfo

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'mousedemo.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
from django.contrib.auth import get_user_model
from django.db import DEFAULT_DB_ALIAS

MOUSEDEMO = str(Path("mousedemo").absolute())
if MOUSEDEMO not in sys.path:
    sys.path.append(MOUSEDEMO)
    
django.setup()

from igvf_mice import models
from igvf_mice.io.converters import instrument_name_to_platform_id

In [2]:
lizs_sheet = "https://woldlab.caltech.edu/nextcloud/index.php/s/eEtjBfDqQFnLpSS/download"


# load ont sequencing sheet from splitseq book

In [3]:
preview_sheet = pandas.read_excel(lizs_sheet, sheet_name="ONT Sequencing", skiprows=1)
preview_sheet

,Experiment,Tech.,Tissue,Sample ID,cDNA Build Date,[cDNA] (ng/uL),Average Length (bp),bp to calculate for loading,ONT Library Replicate ID,Library Build Date,...,Basecalling Config,min Q-score (default 10),Passed bases,Passed Reads,fastq.gz file size (GB),checksum (md5sum) .fastq.gz,Trimmed File,Trimmed Reads,checksum (md5sum) .fastq.gz.1,Notes
0,IGVF003,GF,Main: Cortex + Hippocampus,igvf003_8A_lig-ss_1,2022-12-02,130.0,1150.0,1250.0,ONT001,2022-12-06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Founders,JS,Multiplexed: Heart,igvf003_8A_lig-ss_2,NaT,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,igvf003_8A_lig-ss_3,NaT,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,igvf003_8A_lig-ss_4,NaT,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,igvf003_8A_lig-ss_5,2022-12-02,130.0,1150.0,1250.0,ONT002,2022-12-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,IGVF022,GF,Kidney,igvf022_13A-gc_lig-ss_qc_1,2024-06-12,134.0,1038.0,1138.0,ONT045,2024-06-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,F1s,NaN,NaN,igvf022_13A-gc_lig-ss_p2_1,NaT,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
126,IGVF023,GF,Gonads,igvf023_13A-gc_lig-ss_qc_1,2024-06-12,86.2,1058.0,1158.0,ONT046,2024-06-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
preview_sheet.columns

Index(['Experiment', 'Tech.', 'Tissue', 'Sample ID', 'cDNA Build Date',
       '[cDNA] (ng/uL)', 'Average Length (bp)', 'bp to calculate for loading',
       'ONT Library Replicate ID', 'Library Build Date',
       'Library Molarity (fmol)', 'cDNA Input (ng)', 'cDNA Volume (uL)',
       'H2O Vol. to make up 47uL (uL)', '[Library] (ng/uL)', 'Lib. Vol. (uL)',
       'Total Lib. (ng)', 'Loading Molarity (fmol)', 'Loading Input (ng)',
       'Total Loading Vol. (uL)', 'Sample Loading Vol. (uL)',
       'H2O Vol. to add (uL)', 'Sequencing Date', 'Kit', 'ONT Instrument',
       'Flow Cell Type', 'Flow Cell Chemistry', 'Flow Cell ID', 'Position',
       '# of Pores Dectected', 'Run Length', 'Raw Reads (M)',
       'Estimated Bases (Gb)', 'Estimated N50 (bp)', 'Run Size (GB)',
       'MinKNOW Version', 'Raw Data Type', 'Convert to raw data? ',
       '# of uncoverted files', '# of unconverted reads',
       'final raw data format', '# raw reads', 'raw data file size (GB)',
       'raw data che

# Functions  for parsing the ONT sequencing table

In [5]:
class PlateBoundary(IntEnum):
    START = 0
    PLATE_NAME = 1
    TECHNICIAN = 2

OntSplitseqPlate = namedtuple("OntSplitseqPlate", ["name", "category", "technicians", "start", "end"])
def find_ont_splitseq_plate_boundaries(sheet):
    state = PlateBoundary.START
    plate_name = None
    category = None
    technicians = []
    start_line = None
    end_line = None
    
    for i, row in sheet.iterrows():
        # Don't import incomplete records
        if pandas.notnull(row["Experiment"]) and row["Experiment"] == "F1s" and pandas.isnull(row["cDNA Build Date"]):
            break
        # Every valid record seems to have a Sample ID
        elif pandas.isnull(row["Sample ID"]):
            state = PlateBoundary.START
            end_line = i
            if start_line is not None:
                yield OntSplitseqPlate(plate_name, category, technicians, start_line, end_line)
                plate_name = None
                category = None
                technicians = []
                start_line = None
        elif isinstance(row["Experiment"], str) and row["Experiment"].startswith("IGVF"):
            plate_name = row["Experiment"]
            start_line = i
            state = PlateBoundary.PLATE_NAME
        elif state == PlateBoundary.PLATE_NAME and pandas.notnull(row["Experiment"]) and row["Experiment"] != "Technician":
            category = row["Experiment"]
        
        if pandas.notnull(row["Tech."]):
            technicians.append(row["Tech."])

    if start_line is not None:
        yield OntSplitseqPlate(plate_name, category, technicians, start_line, end_line)
            
list(find_ont_splitseq_plate_boundaries(preview_sheet))
            

[OntSplitseqPlate(name='IGVF003', category='Founders', technicians=['GF', 'JS'], start=0, end=14),
 OntSplitseqPlate(name='IGVF004', category='Founders', technicians=['GF'], start=15, end=34),
 OntSplitseqPlate(name='IGVF005', category='Founders', technicians=[], start=35, end=54),
 OntSplitseqPlate(name='IGVFB01', category='Bridge', technicians=['GF', 'PM'], start=55, end=76),
 OntSplitseqPlate(name='IGVF007', category='Founders', technicians=['GF', 'PM'], start=77, end=82),
 OntSplitseqPlate(name='IGVF008B', category='Founders', technicians=['GF'], start=83, end=85),
 OntSplitseqPlate(name='IGVF009', category=None, technicians=['GF'], start=86, end=88),
 OntSplitseqPlate(name='IGVF010', category='Founders', technicians=['GF'], start=89, end=91),
 OntSplitseqPlate(name='IGVF011', category='Founders', technicians=['GF'], start=92, end=94),
 OntSplitseqPlate(name='IGVF013', category='Founders/F1s', technicians=['GF'], start=95, end=101)]

In [6]:
def get_ont_splitseq_subpool_name(row):
    subpool_re = re.compile("igvf(?P<subpool>[\dB]+_[\d]+[A-Z])")
    sample_id = row["Sample ID"]
    if pandas.isnull(sample_id):
        return
    
    match = subpool_re.match(sample_id)
    if match:
        return match.group("subpool")


# Reading the ONT sequencing table into a form useful for loading

In [7]:
def load_ont_splitseq_sequencing():
    nanopore_samples = {}
    sheet = pandas.read_excel(lizs_sheet, sheet_name="ONT Sequencing", skiprows=1)
    sheet["ONT Instrument"] = sheet["ONT Instrument"].apply(instrument_name_to_platform_id)
    sheet["subpool"] = sheet.apply(get_ont_splitseq_subpool_name, axis=1)

    for ont_plate in find_ont_splitseq_plate_boundaries(sheet):
        for i, row in sheet[ont_plate.start:ont_plate.end].iterrows():        
            tube_id = row["ONT Library Replicate ID"]
            if pandas.isnull(tube_id):
                continue
            elif tube_id in nanopore_samples:
                assert nanopore_samples[tube_id]["cdna_build_date"] == row["cDNA Build Date"]
                assert nanopore_samples[tube_id]["cdna_ng_per_ul"] == row["[cDNA] (ng/uL)"]
                assert nanopore_samples[tube_id]["subpool"] == row["subpool"]
            else:
                nanopore_samples[tube_id] = {
                    "plate": ont_plate.name,
                    "subpool": row["subpool"],
                    "name": tube_id,
                    "sample_id": row["Sample ID"],
                    # nucleic acid extraction attributes
                    "cdna_build_date": row["cDNA Build Date"],
                    "cdna_volume_ul": row["cDNA Volume (uL)"],
                    "cdna_ng_per_ul": row["[cDNA] (ng/uL)"],
                    "average_length": row["Average Length (bp)"],
                    "technician": ",".join(ont_plate.technicians),
                    "passed_qc": True, # Otherwise it wouldn't be in this sheet.
                    # nanopore library attribtes
                    "library_build_date": row["Library Build Date"],
                    "library_volume_ul": row["Lib. Vol. (uL)"],
                    "library_input_ng_per_ul": row["[Library] (ng/uL)"],
                    # sequencing run attributes
                    "sequencing_run_date": row["Sequencing Date"],
                    "sequencing_run_platform": row["ONT Instrument"],
                    "flowcell_kit": row["Kit"],
                    "flowcell_type": row["Flow Cell Type"],
                    "flowcell_id": row["Flow Cell ID"],
                    "sequencing_software": row["Basecaller"],
                    "comments": row["Notes"],
                }


                try:
                    subpool = models.Subpool.objects.get(pk=row["subpool"])
                except models.Subpool.DoesNotExist as e:
                    print("Unable to find subpool {}".format(row["subpool"]))
                    raise e

    return pandas.DataFrame(nanopore_samples.values())

nanopore_df = load_ont_splitseq_sequencing()
nanopore_df

,plate,subpool,name,sample_id,cdna_build_date,cdna_volume_ul,cdna_ng_per_ul,average_length,technician,passed_qc,library_build_date,library_volume_ul,library_input_ng_per_ul,sequencing_run_date,sequencing_run_platform,flowcell_kit,flowcell_type,flowcell_id,sequencing_software,comments
0,IGVF003,003_8A,ONT001,igvf003_8A_lig-ss_1,2022-12-02,1.153846,130.0,1150.0,"GF,JS",True,2022-12-06,15.0,2.34,2022-12-06,gridion,SQK-LSK114,FLO-MIN114,FAU11962,Guppy,NaN
1,IGVF003,003_8A,ONT002,igvf003_8A_lig-ss_5,2022-12-02,1.153846,130.0,1150.0,"GF,JS",True,2022-12-08,15.0,3.82,2022-12-08,gridion,SQK-LSK114,FLO-MIN114,FAU05901,Guppy,NaN
2,IGVF003,003_8A,ONT003,igvf003_8A_lig-ss_10,2022-12-02,1.153846,130.0,1150.0,"GF,JS",True,2022-12-13,15.0,6.10,2022-12-13,gridion,SQK-LSK114,FLO-MIN114,FAV21462,Guppy,NaN
3,IGVF003,003_13A,ONT018,igvf003_13A-gc_lig-ss_qc_1,2023-05-18,3.091286,48.2,1144.0,"GF,JS",True,2023-05-22,15.0,2.90,2023-06-06,minion,SQK-LSK114-XL,FLO-MIN114,FAW82381,Guppy,NaN
4,IGVF004,004_8A,ONT004,igvf004_8A_lig-ss_1,2022-12-15,1.870968,77.5,1113.0,GF,True,2023-01-03,15.0,5.90,2023-01-03,gridion,SQK-LSK114,FLO-MIN114,FAV32642,Guppy,NaN
5,IGVF004,004_8A,ONT005,igvf004_8A_lig-ss_6,2022-12-15,1.870968,77.5,1113.0,GF,True,2023-01-03,15.0,6.02,2023-01-04,gridion,SQK-LSK114,FLO-MIN114,FAV22222,Guppy,NaN
6,IGVF004,004_8A,ONT006,igvf004_8A_lig-ss_7,2022-12-15,1.870968,77.5,1113.0,GF,True,2023-01-03,15.0,6.74,2023-01-04,gridion,SQK-LSK114,FLO-MIN114,FAV32695,Guppy,NaN
7,IGVF004,004_8A,ONT007,igvf004_8A_lig-ss_8,2022-12-15,1.587097,77.5,1113.0,GF,True,2023-01-13,15.0,6.66,2023-01-13,gridion,SQK-LSK114,FLO-MIN114,FAV32912,Guppy,NaN
8,IGVF004,004_8A,ONT008,igvf004_8A_lig-ss_11,2022-12-15,1.677419,77.5,1113.0,GF,True,2023-01-18,15.0,6.24,2023-01-18,gridion,SQK-LSK114,FLO-MIN114,FAW03011,Guppy,NaN
9,IGVF004,004_13A,ONT019,igvf004_13A-gc_lig-ss_qc_1,2023-05-18,2.660194,51.5,1053.0,GF,True,2023-05-22,15.0,2.78,2023-06-09,minion,SQK-LSK114-XL,FLO-MIN114,FAW87892,Guppy,NaN


In [8]:
%debug

ERROR:root:No traceback has been produced, nothing to debug.


In [9]:
list(models.Platform.objects.all())[-1].name

'p2solo'

In [10]:
models.Platform.objects.get(name="p2solo")

<Platform: ONT P2 Solo>

In [11]:
nanopore_df.iloc[0].index

Index(['plate', 'subpool', 'name', 'sample_id', 'cdna_build_date',
       'cdna_volume_ul', 'cdna_ng_per_ul', 'average_length', 'technician',
       'passed_qc', 'library_build_date', 'library_volume_ul',
       'library_input_ng_per_ul', 'sequencing_run_date',
       'sequencing_run_platform', 'flowcell_kit', 'flowcell_type',
       'flowcell_id', 'sequencing_software', 'comments'],
      dtype='object')

In [12]:
models.Subpool.objects.get(pk="013_13A")

<Subpool: 013_13A>

In [13]:
nanopore_df.iloc[3:5].to_dict("list")

{'plate': ['IGVF003', 'IGVF004'],
 'subpool': ['003_13A', '004_8A'],
 'name': ['ONT018', 'ONT004'],
 'sample_id': ['igvf003_13A-gc_lig-ss_qc_1', 'igvf004_8A_lig-ss_1'],
 'cdna_build_date': [Timestamp('2023-05-18 00:00:00'),
  Timestamp('2022-12-15 00:00:00')],
 'cdna_volume_ul': [3.091286307, 1.870967742],
 'cdna_ng_per_ul': [48.2, 77.5],
 'average_length': [1144.0, 1113.0],
 'technician': ['GF,JS', 'GF'],
 'passed_qc': [True, True],
 'library_build_date': [Timestamp('2023-05-22 00:00:00'),
  Timestamp('2023-01-03 00:00:00')],
 'library_volume_ul': [15.0, 15.0],
 'library_input_ng_per_ul': [2.9, 5.9],
 'sequencing_run_date': [Timestamp('2023-06-06 00:00:00'),
  Timestamp('2023-01-03 00:00:00')],
 'sequencing_run_platform': ['minion', 'gridion'],
 'flowcell_kit': ['SQK-LSK114-XL', 'SQK-LSK114'],
 'flowcell_type': ['FLO-MIN114', 'FLO-MIN114'],
 'flowcell_id': ['FAW82381', 'FAV32642'],
 'sequencing_software': ['Guppy', 'Guppy'],
 'comments': [nan, nan]}

# Try comparing the illumina experiment and the ONT sequencing tabs

Here I was trying to figure out why there were sometimes two different concentrations, that sometimes did, and sometimes did not match the source subpool concentration.

It turned out what it was the exome capture experiments had a spearate gene capture concentration, and so for exome capture experiments the ont sequencing concentration should match the gene capture concentration.

In [14]:
records = []
for i, row in nanopore_df.iterrows():
    subpool_name = row["subpool"]
    try:
        subpool = models.Subpool.objects.get(pk=subpool_name)
        
        concentration_pass = subpool.cdna_ng_per_ul == row["cdna_ng_per_ul"]
        gene_pass = subpool.gene_capture_ng_per_ul == row["cdna_ng_per_ul"]
        ont_lt = subpool.cdna_ng_per_ul < row["cdna_ng_per_ul"]
        average_length_pass = subpool.library_average_bp_length == row["average_length"]
        records.append({
            "sublibrary": subpool_name,
            "sample_id": row["sample_id"],
            "tube": row["name"],
            "lib type": subpool.selection_type,
            "conc. =": concentration_pass,
            "gene =": gene_pass,
            #"conc. ont <": ont_lt,
            "subpool cdna ng/ul": subpool.cdna_ng_per_ul,
            "gene cdna ng/ul": subpool.gene_capture_ng_per_ul,
            "ont cdna ng/ul": row["cdna_ng_per_ul"],
            #"length =": average_length_pass,
            "experiment len": subpool.library_average_bp_length,
            "ont len": row["average_length"],
        })
    except models.Subpool.DoesNotExist:
        print(f"Unable to find {subpool_name}")

comparing_libraries = pandas.DataFrame(records)
comparing_libraries

,sublibrary,sample_id,tube,lib type,conc. =,gene =,subpool cdna ng/ul,gene cdna ng/ul,ont cdna ng/ul,experiment len,ont len
0,003_8A,igvf003_8A_lig-ss_1,ONT001,NO,True,False,130.0,NaN,130.0,443.0,1150.0
1,003_8A,igvf003_8A_lig-ss_5,ONT002,NO,True,False,130.0,NaN,130.0,443.0,1150.0
2,003_8A,igvf003_8A_lig-ss_10,ONT003,NO,True,False,130.0,NaN,130.0,443.0,1150.0
3,003_13A,igvf003_13A-gc_lig-ss_qc_1,ONT018,EX,False,True,66.4,48.2,48.2,NaN,1144.0
4,004_8A,igvf004_8A_lig-ss_1,ONT004,NO,True,False,77.5,NaN,77.5,422.0,1113.0
5,004_8A,igvf004_8A_lig-ss_6,ONT005,NO,True,False,77.5,NaN,77.5,422.0,1113.0
6,004_8A,igvf004_8A_lig-ss_7,ONT006,NO,True,False,77.5,NaN,77.5,422.0,1113.0
7,004_8A,igvf004_8A_lig-ss_8,ONT007,NO,True,False,77.5,NaN,77.5,422.0,1113.0
8,004_8A,igvf004_8A_lig-ss_11,ONT008,NO,True,False,77.5,NaN,77.5,422.0,1113.0
9,004_13A,igvf004_13A-gc_lig-ss_qc_1,ONT019,EX,False,True,23.8,51.5,51.5,NaN,1053.0


In [15]:
comparing_libraries[["conc. ="]].sum(axis=0)

conc. =    14
dtype: int64

In [16]:
comparing_libraries[comparing_libraries["conc. ="] == True]

,sublibrary,sample_id,tube,lib type,conc. =,gene =,subpool cdna ng/ul,gene cdna ng/ul,ont cdna ng/ul,experiment len,ont len
0,003_8A,igvf003_8A_lig-ss_1,ONT001,NO,True,False,130.0,NaN,130.0,443.0,1150.0
1,003_8A,igvf003_8A_lig-ss_5,ONT002,NO,True,False,130.0,NaN,130.0,443.0,1150.0
2,003_8A,igvf003_8A_lig-ss_10,ONT003,NO,True,False,130.0,NaN,130.0,443.0,1150.0
4,004_8A,igvf004_8A_lig-ss_1,ONT004,NO,True,False,77.5,NaN,77.5,422.0,1113.0
5,004_8A,igvf004_8A_lig-ss_6,ONT005,NO,True,False,77.5,NaN,77.5,422.0,1113.0
6,004_8A,igvf004_8A_lig-ss_7,ONT006,NO,True,False,77.5,NaN,77.5,422.0,1113.0
7,004_8A,igvf004_8A_lig-ss_8,ONT007,NO,True,False,77.5,NaN,77.5,422.0,1113.0
8,004_8A,igvf004_8A_lig-ss_11,ONT008,NO,True,False,77.5,NaN,77.5,422.0,1113.0
10,005_8A,igvf005_8A_lig-ss_1,ONT009,NO,True,False,38.5,NaN,38.5,438.0,1125.0
11,005_8A,igvf005_8A_lig-ss_7,ONT010,NO,True,False,38.5,NaN,38.5,438.0,1125.0


In [17]:
comparing_libraries[(comparing_libraries["conc. ="] == False) & (comparing_libraries["gene ="] == False)]

,sublibrary,sample_id,tube,lib type,conc. =,gene =,subpool cdna ng/ul,gene cdna ng/ul,ont cdna ng/ul,experiment len,ont len
15,B01_13G,igvfB01_13G-gc_lig-ss_1,ONT012,EX,False,False,48.0,449.0,43.2,444.0,1473.0
16,B01_13G,igvfB01_13G-gc_lig-ss_4,ONT014,EX,False,False,48.0,449.0,43.2,444.0,1473.0
17,B01_13G,igvfB01_13G-gc_lig-ss_7,ONT016,EX,False,False,48.0,449.0,43.2,444.0,1473.0
18,B01_13G,igvfB01_13G-gc_lig-ss_11,ONT021,EX,False,False,48.0,449.0,35.5,444.0,1091.0
27,011_13A,igvf011_13A-gc_lig-ss_qc_1,ONT034,EX,False,False,98.0,32.2,32.5,NaN,1080.0


In [18]:
comparing_libraries[comparing_libraries["sublibrary"] == "003_13A"]

,sublibrary,sample_id,tube,lib type,conc. =,gene =,subpool cdna ng/ul,gene cdna ng/ul,ont cdna ng/ul,experiment len,ont len
3,003_13A,igvf003_13A-gc_lig-ss_qc_1,ONT018,EX,False,True,66.4,48.2,48.2,NaN,1144.0


In [19]:
comparing_libraries.iloc[0]["subpool cdna ng/ul"] < comparing_libraries.iloc[0]["ont cdna ng/ul"]

False

In [20]:
models.Subpool.objects.get(pk="003_8A")

<Subpool: 003_8A>

In [21]:
comparing_libraries.iloc[3]

sublibrary                               003_13A
sample_id             igvf003_13A-gc_lig-ss_qc_1
tube                                      ONT018
lib type                                      EX
conc. =                                    False
gene =                                      True
subpool cdna ng/ul                          66.4
gene cdna ng/ul                             48.2
ont cdna ng/ul                              48.2
experiment len                               NaN
ont len                                   1144.0
Name: 3, dtype: object

# Understanding what the tube id was

Tube ID was later renamed to "ONT Library Replicate ID", it represents the several builds of the nanopore libraries that were then poured into the sequncer.

In [22]:
def view_dupes(name, dupes):
    for key in sorted(dupes):
        if len(dupes[key]) > 1:
            print(name, key, ",".join(dupes[key]))


In [23]:
tube_column_name = "ONT Library Replicate ID"

In [24]:
tubes = {}
subpool = {}
for i, row in preview_sheet.iterrows():
    # skip blank lines
    if numpy.all(pandas.isnull(row)) or row["Experiment"] == "F1s" or pandas.isnull(row[tube_column_name]):
        continue
    subpool_name = get_ont_splitseq_subpool_name(row)
    tubes.setdefault(str(row[tube_column_name]), set()).add(str(subpool_name))
    subpool.setdefault(subpool_name, set()).add(row[tube_column_name])
    
view_dupes("tubes", tubes)
view_dupes("subpool", subpool)


subpool 003_8A ONT002,ONT003,ONT001
subpool 004_8A ONT004,ONT007,ONT008,ONT006,ONT005
subpool 005_13A ONT029,ONT020&026
subpool 005_8A ONT011,ONT009,ONT010
subpool 007_13A ONT030,ONT021&027
subpool 013_13A ONT038,ONT035,ONT036,ONT037
subpool B01_13G ONT014,ONT012,ONT016,ONT021
subpool B01_13H ONT013,ONT015,ONT017


In [25]:
tubes = {}
samples = {}
for i, row in preview_sheet.iterrows():
    # skip blank lines
    if numpy.all(pandas.isnull(row)) or row["Experiment"] == "F1s" or pandas.isnull(row[tube_column_name]):
        continue    
    tubes.setdefault(str(row[tube_column_name]), set()).add(row["Sample ID"])
    samples.setdefault(str(row[tube_column_name]), set()).add(row[tube_column_name])
    
view_dupes("tubes", tubes)
view_dupes("samples", samples)


tubes ONT020&026 igvf005_13A-gc_lig-ss_qc_lid_1,igvf005_13A-gc_lig-ss_p2_1
tubes ONT021&027 igvf007_13A-gc_lig-ss_qc_1,igvf007_13A-gc_lig-ss_p2_1
tubes ONT029 igvf005_13A-gc_lig-ss_p2_2,igvf005_13A-gc_lig-ss_qc_1
tubes ONT030 igvf007_13A-gc_lig-ss_p2_2,igvf007_13A-gc_lig-ss_p2_3,igvf007_13A-gc_lig-ss_qc_2


In [26]:
samples = {}
flowcells = {}
for i, row in preview_sheet.iterrows():
    # skip blank lines
    if numpy.all(pandas.isnull(row)) or row["Experiment"] == "F1s" or pandas.isnull(row["Flow Cell ID"]):
        continue    
    samples.setdefault(row["Sample ID"], set()).add(row["Flow Cell ID"])
    flowcells.setdefault(row["Flow Cell ID"], set()).add(row["Sample ID"])

view_dupes("samples", samples)
view_dupes("flowcells", flowcells)


flowcells PAS39875 igvf013_13A-gc_lig-ss_p2_2,igvf013_13A-gc_lig-ss_p2_2 (all),igvf013_13A-gc_lig-ss_p2_2_3,igvf013_13A-gc_lig-ss_p2_2_2


In [27]:
flowcells["PAS39875"]

{'igvf013_13A-gc_lig-ss_p2_2',
 'igvf013_13A-gc_lig-ss_p2_2 (all)',
 'igvf013_13A-gc_lig-ss_p2_2_2',
 'igvf013_13A-gc_lig-ss_p2_2_3'}

In [28]:
tubes = {}
flowcells = {}
for i, row in preview_sheet.iterrows():
    # skip blank lines
    if numpy.all(pandas.isnull(row)) or row["Experiment"] == "F1s" or pandas.isnull(row["Flow Cell ID"]) or pandas.isnull(row[tube_column_name]):
        continue    
    tubes.setdefault(row[tube_column_name], set()).add(row["Flow Cell ID"])
    flowcells.setdefault(row["Flow Cell ID"], set()).add(row[tube_column_name])
    
view_dupes("tubes", tubes)
view_dupes("flowcells", flowcells)

tubes ONT020&026 FAX29626,PAO02892
tubes ONT021&027 FAX05489,PAO02937
tubes ONT029 PAO02878,FAX09019
tubes ONT030 PAO02852,PAO84050,FAW12158
flowcells PAS39875 ONT038,ONT036,ONT037
